# 回测框架原理和细节

本篇集中说明回测节点的生命周期和报告协议，日常流程见 [基本框架](基本框架.ipynb)。运行状态使用实例身份隔离，语义身份和缓存键的区别见 [Core 原理和细节](../Core/原理和细节.ipynb)。

## BTNode — 回测计算节点

`BTNode` 是回测框架的核心基类，所有回测分析类（如 `IC`、`AccountReport`、`BrinsonModel` 等）都继承自它。`BTNode` 负责计算逻辑，不涉及报告生成。

### 参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"BTNode"` | 节点名称（冻结参数） |

### 核心方法

```python
class BTNode(Node):
    def init_compute(self, path, init_data, context) -> List[DTInitData]:
        """初始化计算。合并多个依赖节点的 DTRange，传递给子节点。"""

    def forward_compute(self, path, fwd_data, context) -> Tuple[List[DTLocalContext], DTLocalContext]:
        """前向传播。将时点列表传递给子节点。"""

    def backward_compute(self, path, bwd_data_list, context, local_context) -> dict:
        """反向传播。接收子节点计算结果，执行回测分析逻辑。子类重写此方法实现具体分析。"""

    def merge_result(self, result_list, context):
        """合并多个进程的计算结果（多进程模式下使用）。默认返回第一个结果。"""
```

### 生命周期详解

```
                   ┌───────────────┐
                   │ init_compute  │  ← 合并到达本节点的初始化请求 DTRange
                   └──────┬────────┘
                          │
                   ┌──────▼────────┐
                   │forward_compute│  ← 将 DTs 传递给子节点
                   └──────┬────────┘
                          │
                   ┌──────▼─────────┐
                   │backward_compute│  ← 收集子节点结果，执行分析
                   └────────────────┘
```

`init_compute` 阶段通过 `context.NodeState` 记录合并后的 `dt_range`，后续子节点可访问此状态获取时点范围。如果节点在依赖链中出现循环引用（`self.InstanceID in path[:-1]`），返回空列表避免无限递归。

## 报告节点与汇总

`ReportNode` 继承 `BTNode`，在计算结果中增加 HTML。参数 `ReportKey` 默认是 `"Report"`。具体报告节点通过具名依赖绑定分析节点，例如 `ICReport(ic_node)`。

`BTReport(report_nodes=[...])` 按依赖顺序汇总；模块结果使用 `"序号-节点名称"` 为键。每个模块可使用自己的 `ReportKey`，汇总键由 `BTReport` 的 `ReportKey` 决定。`BTReport.genOutputReport()` 可拼接已有输出，无需重新执行计算。

`BTNode.merge_result()` 默认只取首个进程的结果，自定义统计节点若需要合并分片，应实现符合其业务语义的合并方法。引擎分片规则见 [计算引擎](../Core/计算引擎.ipynb)。

## 计算与展示分离

算子产生逐期因子，分析节点汇总统计结果，报告节点将统计结果转换为 HTML。只需要数值数据时，可直接把分析节点作为引擎目标，减少绘图开销。

扩展节点时用 `__QS_DepClass__` 声明具名依赖，业务参数通过 `__QS_ArgClass__` 声明，避免将影响结果的配置仅写入私有属性。节点模板、复制和输入校验见 [计算图框架](../Core/计算图框架.ipynb)。